### The experiment setup

We compare two models:

- Plain CNN: deep, stacked layers
- Residual CNN: same depth, skip connections

Rules
- same dataset
- same optimizer
- same learning rate
- same epochs

If one trains better, architecture is the reason.

### What “deep” means here

Depth means repeated application of functions:

$y = f_n(f_{n-1}(\dots f_1(x)))$


As $n$ increases:
- optimization gets harder
- learning identity becomes difficult

We will push depth just enough to expose this.

### Plain deep CNN

Conceptually, each block does:

$y = f(x)$


Stacked many times.

#### Code

In [97]:
from torchvision import datasets, transforms

transform = transforms.ToTensor()

train_data = datasets.MNIST(
    train=True,
    transform=transform,
    root="../M3 CNNs/data",
    download=False
)

test_data = datasets.MNIST(
    train=False,
    transform=transform,
    root="../M3 CNNs/data",
    download=False
)

In [98]:
import torch.nn as nn

class PlainCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.ReLU(),
            
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10)
        )
        
    def forward(self, x):
        return self.net(x)

#### What is nn.Module?

Every neural network in PyTorch:

- inherits from nn.Module
- defines layers in __init__()
- defines computation in forward()

PyTorch tracks:
- parameters
- gradients
- device placement

through this base class.

`nn.AdaptiveAvgPool2d(output_size)` means:

> “Whatever the input spatial size is, produce exactly this output size.”

So,

It will convert any [C, H, W] into [C, 1, 1].

Key point
- every layer must learn something useful
- doing “nothing” is not easy

Example

If input is:

`[batch, 32, 32, 32]`


After:

`nn.AdaptiveAvgPool2d(1)`


It becomes:

`[batch, 32, 1, 1]`


It averages across all spatial positions.

### Residual block

Residual block implements:

$y = x + g(x)$


Where $g(x)$ is a small CNN.

#### Code

In [99]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1)
        )
        
    def forward(self, x):
        return x + self.block(x)
        # x + g(x)

- input x flows unchanged
- block only learns corrections

### Residual CNN

In [100]:
class ResidualCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.ReLU()
        )
        
        self.res_blocks = nn.Sequential(
            ResidualBlock(32),
            ResidualBlock(32),
            ResidualBlock(32),
            ResidualBlock(32)
        )
        
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10)
        )
        
    def forward(self, x):
        x = self.stem(x)
        x = self.res_blocks(x)
        return self.head(x)

Depth ≈ same as plain CNN. <br>
Only difference = skip connections.

### Why these two are NOT equivalent

Plain CNN learns:

$y = f_4(f_3(f_2(f_1(x))))$


Residual CNN learns:

$y = x + g_1(x) + g_2(x) + g_3(x) + g_4(x)$


This second form:
- biases network toward identity
- allows gradual refinement
- stabilizes gradients

In [101]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_data,
    batch_size=128,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_data,
    batch_size=128,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [102]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

loss_fn = nn.CrossEntropyLoss()

In [103]:
def train_model(cnn_model):

    model = cnn_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    epochs = 10
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            preds = model(images)
            loss = loss_fn(preds, labels)
            
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            
            epoch_loss += loss.item()
            
        epoch_loss /= len(train_loader)
        
        print(f"Epoch: {epoch} | Avg loss: {epoch_loss:.4f}")
        
        
    correct = 0
    total = 0
    model.eval()
    
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        preds = model(images)  
        predicted = preds.argmax(dim=1)
        
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
    accuracy = correct / total
    print("Accuracy: ", accuracy)

In [104]:
print("Plain CNN:\n")
train_model(PlainCNN)

print("\nResidual CNN:\n")
train_model(ResidualCNN)

Plain CNN:

Epoch: 0 | Avg loss: 1.6226
Epoch: 1 | Avg loss: 0.9545
Epoch: 2 | Avg loss: 0.6540
Epoch: 3 | Avg loss: 0.5065
Epoch: 4 | Avg loss: 0.4213
Epoch: 5 | Avg loss: 0.3786
Epoch: 6 | Avg loss: 0.3262
Epoch: 7 | Avg loss: 0.2977
Epoch: 8 | Avg loss: 0.2667
Epoch: 9 | Avg loss: 0.2515
Accuracy:  0.9246

Residual CNN:

Epoch: 0 | Avg loss: 0.9737
Epoch: 1 | Avg loss: 0.2965
Epoch: 2 | Avg loss: 0.1921
Epoch: 3 | Avg loss: 0.1468
Epoch: 4 | Avg loss: 0.1115
Epoch: 5 | Avg loss: 0.0886
Epoch: 6 | Avg loss: 0.0791
Epoch: 7 | Avg loss: 0.0667
Epoch: 8 | Avg loss: 0.0602
Epoch: 9 | Avg loss: 0.0532
Accuracy:  0.9868


### Expected Output

Plain CNN

- loss decreases slowly
- may plateau early
- sensitive to learning rate
- deeper = worse

Residual CNN

- loss decreases smoothly
- trains faster
- deeper ≠ worse
- stable gradients

If you don’t see this difference:
- depth is too shallow
- or dataset too easy